In [13]:
import pandas as pd
import pyodbc

# ========== 1️⃣ 基本信息 ==========
server = "dongjing-sql-server.database.windows.net"
database = "insurance_db"
username = "sqladmin"
password = "YourStrongPassword123!"
csv_path = "data/clean/Insurance_claims_data_cleaned.csv"

# Azure SQL 连接
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server=tcp:{server},1433;"
    f"Database={database};"
    f"Uid={username};"
    f"Pwd={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("✅ Connected to Azure SQL Database")




✅ Connected to Azure SQL Database


In [14]:
from math import ceil

cursor.fast_executemany = True   # 打开批量优化

def batched(lst, size=1000):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

In [15]:
customers = df[['customer_id','customer_age']].drop_duplicates('customer_id').reset_index(drop=True)
customers['age_group'] = pd.cut(customers['customer_age'],
                                bins=[0,25,35,45,55,100],
                                labels=['<25','26-35','36-45','46-55','55+'])

params = [(int(r.customer_id), float(r.customer_age), r.age_group) for _, r in customers.iterrows()]

for chunk in batched(params, 1000):
    cursor.executemany("""
        INSERT INTO Dim_Customer (customer_id, customer_age, age_group)
        VALUES (?, ?, ?)
    """, chunk)
    conn.commit()
print(f"✅ Inserted {len(params)} rows into Dim_Customer")

✅ Inserted 30349 rows into Dim_Customer


In [16]:
fueltypes = df[['fuel_type']].drop_duplicates().reset_index(drop=True)
fueltypes['fuel_category'] = fueltypes['fuel_type'].apply(
    lambda x: 'EV' if str(x).lower() in ['electric','hybrid'] else 'Combustion'
)
params = [(r.fuel_type, r.fuel_category) for _, r in fueltypes.iterrows()]

for chunk in batched(params, 1000):
    cursor.executemany("""
        INSERT INTO Dim_FuelType (fuel_type_name, fuel_category)
        VALUES (?, ?)
    """, chunk)
    conn.commit()
print(f"✅ Inserted {len(params)} rows into Dim_FuelType")


✅ Inserted 3 rows into Dim_FuelType


In [17]:
dates = pd.date_range(start="2020-01-01", end="2025-12-31")
time_df = pd.DataFrame({
    'full_date': dates,
    'year': dates.year,
    'quarter': dates.quarter,
    'month': dates.month,
    'week': dates.isocalendar().week.astype(int),
    'day': dates.day
})
params = [(r.full_date, int(r.year), int(r.quarter), int(r.month), int(r.week), int(r.day))
          for _, r in time_df.iterrows()]

for chunk in batched(params, 2000):
    cursor.executemany("""
        INSERT INTO Dim_Time (full_date, year, quarter, month, week, day)
        VALUES (?, ?, ?, ?, ?, ?)
    """, chunk)
    conn.commit()
print(f"✅ Inserted {len(params)} rows into Dim_Time")


✅ Inserted 2192 rows into Dim_Time


In [18]:
vehicles = df[['vehicle_age','segment','model','fuel_type','engine_type',
               'max_torque','max_power','displacement','cylinder',
               'transmission_type','steering_type','turning_radius',
               'length','width','gross_weight','airbags','is_esc',
               'is_parking_camera','is_brake_assist','ncap_rating']].drop_duplicates().reset_index(drop=True)

fuel_map = pd.read_sql("SELECT fuel_type_id, fuel_type_name FROM Dim_FuelType", conn)
fuel_dict = dict(zip(fuel_map['fuel_type_name'], fuel_map['fuel_type_id']))
vehicles['fuel_type_id'] = vehicles['fuel_type'].map(fuel_dict)

# 如出现无法映射的 fuel_type，先丢弃（或补默认）
vehicles = vehicles.dropna(subset=['fuel_type_id'])

def b(x): return 1 if str(x).lower() == 'yes' else 0

params = [(
    float(r.vehicle_age), r.segment, r.model, int(r.fuel_type_id), r.engine_type, r.max_torque, r.max_power,
    int(r.displacement), int(r.cylinder), r.transmission_type, r.steering_type, float(r.turning_radius),
    int(r.length), int(r.width), int(r.gross_weight), int(r.airbags), b(r.is_esc), b(r.is_parking_camera),
    b(r.is_brake_assist), int(r.ncap_rating)
) for _, r in vehicles.iterrows()]

for chunk in batched(params, 1000):
    cursor.executemany("""
        INSERT INTO Dim_Vehicle (
          vehicle_age, segment, model, fuel_type_id, engine_type, max_torque, max_power,
          displacement, cylinder, transmission_type, steering_type, turning_radius,
          length, width, gross_weight, airbags, is_esc, is_parking_camera, is_brake_assist, ncap_rating
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, chunk)
    conn.commit()
print(f"✅ Inserted {len(params)} rows into Dim_Vehicle")


C:\Users\86133\AppData\Local\Temp\ipykernel_18816\3393964659.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fuel_map = pd.read_sql("SELECT fuel_type_id, fuel_type_name FROM Dim_FuelType", conn)


✅ Inserted 281 rows into Dim_Vehicle


In [19]:
# --- Region (批量插入) ---
regions = df[['region_code','region_density']].drop_duplicates().reset_index(drop=True)
regions['region_tier'] = pd.cut(regions['region_density'],
                                bins=[0,10000,20000,40000],
                                labels=['Low','Medium','High'])

cursor.fast_executemany = True
params = [(r.region_code, int(r.region_density), r.region_tier) for _, r in regions.iterrows()]

for i in range(0, len(params), 1000):
    chunk = params[i:i+1000]
    cursor.executemany("""
        INSERT INTO Dim_Region (region_code, region_density, region_tier)
        VALUES (?, ?, ?)
    """, chunk)
    conn.commit()

print(f"✅ Inserted {len(params)} rows into Dim_Region")


✅ Inserted 22 rows into Dim_Region


In [23]:
# 1) 读取维度映射（region 正常；vehicle 压成唯一）
region_map = pd.read_sql("SELECT region_id, region_code FROM Dim_Region;", conn)

# 关键：同一 model 只保留一个 vehicle_id（选最小或最大都行）
vehicle_map = pd.read_sql("""
    SELECT MIN(vehicle_id) AS vehicle_id, model
    FROM Dim_Vehicle
    GROUP BY model
""", conn)

# 2) 构建事实表数据（先确保 policy_id 唯一）
df_fact = (df.drop_duplicates(subset=['policy_id'])
             .merge(region_map,  on='region_code', how='left')
             .merge(vehicle_map, on='model',       how='left'))

# 3) 丢弃无法匹配外键的行（可先导出排查）
bad = df_fact[df_fact['region_id'].isna() | df_fact['vehicle_id'].isna()]
print("⚠️ 无法匹配外键的行数:", len(bad))
# bad.to_csv("data/bad_fact_rows.csv", index=False)

df_fact = df_fact.dropna(subset=['region_id','vehicle_id'])

# 4) 批量插入
cursor.fast_executemany = True
batch_size = 1000
for i in range(0, len(params), batch_size):
    chunk = params[i:i+batch_size]
    # 在 VALUES 后面加 WHERE NOT EXISTS，最后再带一个 policy_id 参数做查重
    chunk2 = [p + (p[0],) for p in chunk]  # 把 policy_id 再放到第7个参数
    cursor.executemany("""
        INSERT INTO Fact_Policy (
          policy_id, subscription_length, claim_status,
          customer_id, vehicle_id, region_id
        )
        SELECT ?,?,?,?,?,?
        WHERE NOT EXISTS (SELECT 1 FROM Fact_Policy WHERE policy_id = ?)
    """, chunk2)
    conn.commit()


cursor.execute("ALTER TABLE Fact_Policy WITH CHECK CHECK CONSTRAINT ALL;")
conn.commit()
print(f"✅ Fact_Policy inserted: {len(params):,} rows")



C:\Users\86133\AppData\Local\Temp\ipykernel_18816\711857731.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  region_map = pd.read_sql("SELECT region_id, region_code FROM Dim_Region;", conn)
C:\Users\86133\AppData\Local\Temp\ipykernel_18816\711857731.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vehicle_map = pd.read_sql("""


⚠️ 无法匹配外键的行数: 0
✅ Fact_Policy inserted: 58,592 rows


In [8]:
cursor.close()
conn.close()
print("🎯 All data successfully imported & Azure SQL connection closed cleanly!")

🎯 All data successfully imported & Azure SQL connection closed cleanly!
